## Környezet ellenőrzése

In [ ]:
!conda info --envs

## Notebook tartalomjegyzéke

Felhasznált adathalmaz: **Online Retail II UCI**
https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data  


0\. Adatbetöltés és parquet konverzió

1\. Adattisztítás és alap EDA SQLite segítségével

2\. RFM Feature Engineering: alapmutatók kiszámítása.

3\. (Statisztikai) outlier-kezelés és skálázás: (Pl. StandardScaler vagy MinMaxScaler a K-means előtt, mert a távolság alapú algoritmusok érzékenyek a léptékre).

4\. K-means klaszterezés: Szegmentálás és a szegmensek profilozása.

5\. Kiterjesztett EDA: A klaszterek vizualizálása (pl. Snake plot vagy Heatmap).

6\. XGBoost Célváltozó tervezés: Címkézés (pl. Churn: 0 vagy 1) egy adott időpont alapján.

7\. Modellezés és SHAP: Tanítás, tesztelés és a modell magyarázhatósága (melyik RFM faktor a legfontosabb a churnnél?).

8\. Üzleti kiértékelés: Mit kezdjünk az egyes csoportokkal? (Pl. "Hűségeseknek kedvezmény", "Lemorzsolódóknak re-engagement kampány").

9\. Export: Streamlit-hez vagy egyéb BI eszközhöz.

## Műszaki és verziókezelési megjegyzések

**Adatformátum (Parquet):** A projekt során a nyers CSV adatokat az első lépésben Apache Parquet formátumba transzformálom és a data/processed/ mappában tárolom az alábbi, data engineeringben ismert előnyök miatt:


Hatékonyság: Az oszlop-alapú tárolás jelentősen gyorsabb beolvasást és kisebb memóriahasználatot tesz lehetővé a feature engineering során.


Típusbiztonság: A Parquet megőrzi a sémát (pl. InvoiceDate datetime vagy Customer ID integer típusát), így elkerülhetők a CSV-re jellemző ismételt típuskonverziós hibák.

**Verziókezelés (nbstripout):** A notebook tisztán tartása érdekében ``nbstripout`` használatával dolgozom. Ez a git-filter biztosítja, hogy a távoli repóba (GitHub) kizárólag a forráskód kerüljön be, a futtatási metaadatok és nagyméretű vizuális kimenetek nélkül. Ez elősegíti a tiszta code review folyamatot és megakadályozza a bináris szemét felhalmozódását a verziótörténetben.

---
## 0. Adatbetöltés és Parquet-konverzió

A nyers adathalmaz betöltése során elsődlegesen az automatizált megoldásra törekszünk. Mivel azonban a specifikus UCI API korlátokba ütközött, a kód egy robusztus ellenőrző logikát használ: ha a nyers adatok hiányoznak, pontos instrukciókat ad a manuális beszerzéshez, majd elvégzi a Parquet konverziót az optimális további feldolgozáshoz.

A `0.2` cella idempotens: ha a tisztított Parquet fájl már létezik, automatikusan kihagyja az ismételt letöltést és konverziót.

In [ ]:
# ============================================================
# 0.1 – Könyvtárak, konstansok és adat-ellenőrzés
# ============================================================
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Útvonalak beállítása
PROJECT_ROOT = Path(".").resolve()
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IMAGES_DIR    = PROJECT_ROOT / "assets" / "images"
MODELS_DIR    = PROJECT_ROOT / "models"

# Mappastruktúra létrehozása
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Fájlnevek
RAW_FILE = RAW_DIR / "online_retail_II.csv"
PARQUET_OUT = PROCESSED_DIR / "online_retail_raw.parquet"

print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"RAW_FILE:      {RAW_FILE}")
print(f"PARQUET_OUT:   {PARQUET_OUT}\n")

# --- Klónozás utáni állapot ellenőrzése ---
if not PARQUET_OUT.exists() and not RAW_FILE.exists():
    error_msg = (
        "\n" + "="*80 + "\n"
        "HIÁNYZÓ ADATHALMAZ!\n"
        "A hivatalos UCI API nem támogatja ezt a specifikus adathalmazt, így manuális letöltés szükséges.\n\n"
        "Kérlek, kövesd az alábbi lépéseket a folytatáshoz:\n"
        "1. Töltsd le a zip fájlt a Kaggle-ről:\n"
        "   https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data\n"
        "2. Csomagold ki, és az 'online_retail_II.csv' fájlt helyezd el ide:\n"
        f"   {RAW_FILE}\n"
        "3. Futtasd újra ezt a cellát!\n"
        + "="*80 + "\n"
    )
    print(error_msg)
    raise FileNotFoundError("A nyers adathalmaz nem található. Kövesd a fenti utasításokat!")
else:
    print("Fájlrendszer ellenőrizve: a szükséges adatfájlok rendelkezésre állnak!")

In [ ]:
# ============================================================
# 0.2 – Nyers CSV betöltése és Parquet-be konvertálása
# ============================================================

if PARQUET_OUT.exists():
    print(f"Parquet már létezik, kihagyjuk a konverziót: {PARQUET_OUT}")
else:
    dtype_map = {
        "Invoice":     "string",
        "StockCode":   "string",
        "Description": "string",
        "Quantity":    "float64",
        "Price":       "float64",
        "Customer ID": "float64",
        "Country":     "string",
    }
    parse_dates = ["InvoiceDate"]

    print(f"CSV fájl betöltése innen: {RAW_FILE} ... (ez eltarthat egy percig)")
    df_raw = pd.read_csv(
        RAW_FILE,
        dtype=dtype_map,
        parse_dates=parse_dates,
        encoding="utf-8",
    )
    
    print(f"Összesített sorok (nyers): {len(df_raw):,}")

    # Customer ID: float -> nullable Int64 (megőrzi a NaN-okat is)
    df_raw["Customer ID"] = df_raw["Customer ID"].astype("Int64")

    # Parquet mentés
    df_raw.to_parquet(PARQUET_OUT, compression="snappy", index=False)

    size_mb = PARQUET_OUT.stat().st_size / 1_048_576
    print(f"\nParquet mentve: {PARQUET_OUT}")
    print(f"Fájlméret:      {size_mb:.1f} MB")
    print(f"Sorok:          {len(df_raw):,} | Oszlopok: {df_raw.shape[1]}")
    print(f"\nSéma:\n{df_raw.dtypes}")

## 1. Adattisztítás

### 1.1. Első lépések

A Customer Lifetime Value (CLV) modellezés és az RFM szegmentáció alapfeltétele, hogy az adatokat vásárlói szinten tudjuk aggregálni. Ennek megfelelően az alábbi tisztítási lépéseket végezzük el:

- **Anonim tranzakciók eltávolítása:** A hiányzó `Customer ID`-val rendelkező sorok kötelezően eldobandók.
- **Azonosítók típuskonverziója:** A `Customer ID`-t numerikus formátumból (Int64) string (object) típusra alakítjuk, hogy elkerüljük az EDA során a fals statisztikákat, és megelőzzük a későbbi gépi tanulási modelleknél (K-means, XGBoost) az adatszivárgást (data leakage).
- **Érvénytelen értékek szűrése:** A 0 vagy negatív `Price` értékkel rendelkező sorok eltávolítása.
- **Duplikátumok szűrése:** Az azonos tartalmú, ismétlődő sorok törlése.
- **Adminisztratív és nem termék kódok eltávolítása:** A nyers adatokon végzett előzetes **SQLite-os adatfeltárás** során kiderült, hogy az adathalmazban jelentős számú, torzító hatású adminisztratív bejegyzés található (pl. `POST`, `M`, `BANK CHARGES`, `AMAZONFEE`). Ezek mellé soroljuk a `C2` (Carriage) kódot is, amely egy kiugróan magas árú, fix tengerentúli fuvardíjra utal. Mivel ezek nem kézzelfogható termékek (és gyakran extrém negatív korrekciós értékeket képviselnek), teljesen kiszűrjük őket, hogy ne torzítsák a termékalapú kosárelemzést és a CLV-t.
- **Fejlesztői teszt bejegyzések szűrése:** Szintén az SQLite-os lekérdezések buktatták le a rendszerben maradt teszt tranzakciókat (pl. `TEST001`, `TEST002` cikkszámok, vagy `test` leírások). Mivel ezek irreális árakkal és mennyiségekkel rontanák a statisztikákat, a `StockCode` és a `Description` oszlopok együttes vizsgálatával az összeset eldobjuk.

*Megjegyzés: A sztornó/visszaküldött tételeket ('C' prefix az Invoice oszlopban) ebben a fázisban szándékosan megtartjuk, mivel ezek a későbbiekben negatív feature-ként szolgálnak a vásárlói érték (CLV) számításánál.*

In [ ]:
# ============================================================
# 1.1 Adattisztítás
# ============================================================
import pandas as pd

print("Adattisztítás megkezdése...\n")

# 1.1.1. Nyers Parquet betöltése
df = pd.read_parquet(PARQUET_OUT)
eredeti_sorok_szama = len(df)
print(f"Kiinduló sorok száma: {eredeti_sorok_szama:,}")
print("-" * 50)

# 1.1.2. Hiányzó Customer ID-k eldobása
df = df.dropna(subset=["Customer ID"])
id_nelkuli_sorok = eredeti_sorok_szama - len(df)
print(f"✔️ Nincsenek anonim vásárlók. (Eldobva: {id_nelkuli_sorok:,} sor)")

# Opcionális, de ajánlott: Customer ID stringgé alakítása
df["Customer ID"] = df["Customer ID"].astype(str)
print("✔️ A Customer ID biztonságos string formátumban van.")

# 1.1.3. Érvénytelen árak szűrése (Price <= 0)
ervenytelen_ar_maszk = df["Price"] <= 0
ervenytelen_ar_sorok = ervenytelen_ar_maszk.sum()
df = df[~ervenytelen_ar_maszk]
print(f"✔️ Az érvénytelen (0 vagy negatív) árak eltűntek. (Eldobva: {ervenytelen_ar_sorok:,} sor)")

# --- Sztornók megtartása ---
print("✔️ A sztornók szándékosan megmaradnak a feature engineeringhez (Quantity <= 0 szándékosan nincs szűrve).")

# 1.1.4. Nem termék jellegű tranzakciók kiszűrése
nem_termek_maszk = (
    df["StockCode"].astype(str).str.replace(" ", "").str.isalpha() | 
    (df["StockCode"].astype(str).str.upper() == "C2")
)
nem_termek_sorok = nem_termek_maszk.sum()
df = df[~nem_termek_maszk]
print(f"✔️ A 'BANK CHARGES' jellegű és 'C2' fuvardíj kódok célzottan eldobásra kerültek. (Eldobva: {nem_termek_sorok:,} sor)")

# 1.1.5. Teszt bejegyzések eltávolítása
teszt_maszk = (
    df["Description"].astype(str).str.contains("TEST", case=False, na=False) |
    df["StockCode"].astype(str).str.contains("TEST", case=False, na=False)
)
teszt_sorok = teszt_maszk.sum()
df = df[~teszt_maszk]
print(f"✔️ A rejtett, üres leírású tesztkódok is fennakadtak a szűrőn. (Eldobva: {teszt_sorok:,} sor)")

# 1.1.6. Duplikátumok eldobása
duplikatum_maszk = df.duplicated()
duplikatum_sorok = duplikatum_maszk.sum()
df = df[~duplikatum_maszk]
print(f"✔️ A duplikátumok eltávolítva. (Eldobva: {duplikatum_sorok:,} sor)")

# 1.1.7. Tisztított adatok mentése checkpointként
CLEANED_PARQUET = PROCESSED_DIR / "online_retail_cleaned.parquet"
df.to_parquet(CLEANED_PARQUET, compression="snappy", index=False)

vegleges_sorok_szama = len(df)
print("-" * 50)
print(f"Adattisztítás eredménye:")
print(f"Megmaradt tiszta sorok száma: {vegleges_sorok_szama:,} "
      f"(az eredeti {vegleges_sorok_szama/eredeti_sorok_szama*100:.1f}%-a)")
print(f"Tisztított adatok mentve: {CLEANED_PARQUET}")

### 1.2. Ellenőrzés 1: Adatszerkezet és típusok
A tisztítási lépések befejeztével elengedhetetlen ellenőrizni az adathalmaz technikai állapotát. A `df.info()` kimenetével az alábbiakat validáljuk a Feature Engineering megkezdése előtt:
* **Nincsenek hiányzó értékek:** Minden oszlopban hiánytalan a `Non-Null Count` (az "anonim" vásárlásokat sikeresen kidobtuk).
* **Megfelelő adattípusok:** Az `InvoiceDate` helyesen `datetime` típusú, a `Customer ID` pedig konzisztens szöveges (string/object) formátumú.
* **Memóriahatékonyság:** A felesleges és hibás sorok eltávolításával optimalizáltuk a memóriahasználatot, ami a későbbi gépi tanulási modellek (K-means, XGBoost) illesztésénél kritikus lesz.

In [ ]:
# ============================================================
# 1.2 "QA" ellenőrzés 1/2
# ============================================================
df.info()

### 1.3. Ellenőrzés 2: Statisztikai érvényesség és üzleti logika
Míg a kezdeti EDA során a `describe()` a hibák felfedezését szolgálta, itt már **minőségbiztosítási (QA)** szerepe van. Az alábbi kimenet bizonyítja, hogy a tisztítási logikánk sikeres volt:
* **Érvényes árak:** A `Price` oszlop minimum értéke szigorúan **> 0** (nincsenek ingyenes vagy negatív árú, adminisztratív tételek).
* **Reális mennyiségek:** A `Quantity` oszlop extrém, rendszerhibából fakadó kilengései (pl. +/- 80 000 darab) eltűntek, a maximum és minimum értékek a beállított küszöbértéken belül maradtak.
* **Adatszivárgás megelőzése:** Nincsenek olyan anomáliák, amik a későbbi aggregált RFM metrikákat (pl. átlagos kosárérték) irreális irányba torzítanák.

In [ ]:
# ============================================================
# 1.3 "QA" ellenőrzés 2/2
# ============================================================
df.describe()

### 1.4. Technikai outlierek és anomáliák szűrése (Feature Engineering előtt)

Mielőtt rátérnénk a vásárlói érték (RFM és CLV) kiszámítására, el kell távolítanunk a rendszerben maradt **technikai anomáliákat**. Fontos: itt még nem a statisztikai outliereket (pl. a kiugróan sokat költő VIP vevőket) kezeljük, hanem azokat a tranzakciókat, amelyek **torzítanák vagy logikailag ellehetetlenítenék** a vásárlói profilalkotást.

Ebben a lépésben az alábbi "zajokat" szűrjük:
1. **"Árva" visszaküldések és negatív élettartam-értékű (CLV) ügyfelek:** Előfordulnak olyan ügyfelek, akiknek a vizsgált időszakban több a sztornója, mint a vásárlása (tehát a nettó költésük $\le 0$). Ennek tipikus oka, hogy a tényleges vásárlás az adathalmaz kezdete (2009. dec. 1.) *előtt* történt. Mivel róluk nem tudunk valós vásárlói értéket (Monetary) számolni, az ilyen ügyfelek összes sorát kötelezően eldobjuk.
2. **Fizikailag lehetetlen tranzakciók (Extrém Quantity):** Az adatok előzetes vizsgálata kimutatta, hogy léteznek +/- 80 000 darabos tételek. Egy ajándéktárgy-nagykereskedésben (B2B) a pár ezer darabos rendelés még reális, de a 10 000 feletti kiugrások szinte kivétel nélkül azonnal sztornózott rendszerhibák vagy adminisztrációs tesztek. Ezeket a sorokat eltávolítjuk, hogy ne torzítsák a kosárméret-statisztikákat.

In [ ]:
# ============================================================
# 1.4. - Technikai outlierek, "árva" sztornók és hibák szűrése
# ============================================================
df = pd.read_parquet(CLEANED_PARQUET)
print("Technikai outlierek és anomáliák szűrése...")
eredeti_sorok = len(df)

# 1.4.1. Tranzakció nettó értékének (LineTotal) kiszámítása
df['LineTotal'] = df['Quantity'] * df['Price']

# 1.4.2. Manuális korrekciók ("Adjustment") szűrése a Description alapján
# A felfedező elemzés során találtunk "Adjustment by Peter..." jellegű sorokat
adjustment_mask = df['Description'].str.contains('ADJUSTMENT', case=False, na=False)
df = df[~adjustment_mask]

# 1.4.3. Ügyfélszintű aggregáció: mekkora az adott ügyfél teljes nettó költése az időszakban?
customer_revenue = df.groupby('Customer ID')['LineTotal'].sum().reset_index()

# Azonosítjuk azokat az ügyfeleket, akiknek a teljes egyenlege nulla vagy negatív
invalid_customers = customer_revenue[customer_revenue['LineTotal'] <= 0]['Customer ID']
df = df[~df['Customer ID'].isin(invalid_customers)]

# 1.4.4. Extrém darabszámok (Quantity) szűrése (pl. +/- 80 ezer darabos rendszerhibák)
Q_THRESHOLD = 10000
extreme_quantity_mask = df['Quantity'].abs() > Q_THRESHOLD
df = df[~extreme_quantity_mask]

vegleges_sorok = len(df)
print(f"✔️ Manuális korrekciók, negatív CLV-jű ügyfelek és >{Q_THRESHOLD} db-os anomáliák eltávolítva.")
print(f"Elemzésre (RFM) kész sorok száma: {vegleges_sorok:,} "
      f"(az eredeti nyers adathalmaz kb. {vegleges_sorok/1067371*100:.1f}%-a megmaradt)\n")

# Tisztított, RFM-re kész adatok mentése memóriahatékony Parquet formátumban
READY_FOR_RFM_PARQUET = PROCESSED_DIR / "online_retail_ready_for_rfm.parquet"
df.to_parquet(READY_FOR_RFM_PARQUET, compression="snappy", index=False)
print(f"Adathalmaz mentve a következő fázishoz: {READY_FOR_RFM_PARQUET}")

## 2. Feature Engineering és ML Keretrendszer (Time-Window)

A klasszikus prediktív Customer Lifetime Value (CLV) modellezés legkritikusabb pontja az **adatszivárgás (data leakage) megelőzése**. Annak érdekében, hogy egy valós üzleti szituációt szimuláljunk (ahol a múltbeli adatokból próbáljuk megjósolni a jövőt), az adathalmazt egy időbeli vágási pont (Cutoff Date) mentén **két szigorúan elkülönülő ablakra bontjuk**:

1. **Megfigyelési ablak (X - Observation Window):** Az adathalmaz kezdete és a vágási pont közötti időszak. Kizárólag ezekből az adatokból számítjuk ki az ügyfelek RFM (Recency, Frequency, Monetary) és egyéb viselkedési mutatóit. A gépi tanulási modell csak ezt fogja látni.
2. **Célablak (y - Target Window):** A vágási ponttól az adathalmaz végéig tartó időszak (kb. az utolsó 90 nap). Itt számoljuk ki az ügyfelek tényleges értékét (a célváltozót), amit a modellnek majd prediktálnia kell.

*Kiválasztott vágási pont:* **2011. szeptember 9.** (Mivel az adathalmaz 2011. december 9-én ér véget, ez pontosan egy 90 napos (negyedéves) előrejelzési ablakot biztosít, ami iparági sztenderd a B2B/B2C szegmentációban).

In [ ]:
# ============================================================
# 2.1 - Időablakok (Time-Windows) felállítása és szétválasztása
# ============================================================

# Ha a kernelt újraindítanánk, betöltjük a tiszta adatokat:
# df = pd.read_parquet(READY_FOR_RFM_PARQUET)

print("Időablakok szétválasztása az adatszivárgás elkerülése érdekében...")

# Vágási pont (Cutoff date) definiálása: pontosan 90 nappal az adathalmaz vége előtt
CUTOFF_DATE = pd.to_datetime('2011-09-09')

# 2.1. Megfigyelési ablak (X feature-ök alapja)
df_obs = df[df['InvoiceDate'] < CUTOFF_DATE].copy()

# 2.2. Célablak (y célváltozó alapja)
df_target = df[df['InvoiceDate'] >= CUTOFF_DATE].copy()

print("-" * 50)
print(f"Teljes vizsgált időszak:  {df['InvoiceDate'].min().date()}  --->  {df['InvoiceDate'].max().date()}")
print(f"Vágási pont (Cutoff):     {CUTOFF_DATE.date()}")
print("-" * 50)
print(f"Megfigyelési ablak (X):   {len(df_obs):,} sor")
print(f"Célablak (y):             {len(df_target):,} sor")

# 2.3 Ellenőrizzük, hogy hány olyan ügyfél van a megfigyelési ablakban, akit érdemes vizsgálni
valos_ugyfelek_szama = df_obs['Customer ID'].nunique()
print(f"\\nModellezhető egyedi ügyfelek száma a megfigyelési ablakban: {valos_ugyfelek_szama:,}")

### 2.2 Kiterjesztett RFM Feature Engineering (csak az X ablakon)

A megfigyelési ablak (`df_obs`) felhasználásával kiszámítjuk a vásárlók egyedi profilját leíró mutatókat. A hagyományos RFM modellt **kiterjesztjük a visszaküldési (return) metrikákkal**, mivel a magas sztornóarány kritikus indikátora a jövőbeli lemorzsolódásnak (churn) és a csökkenő élettartam-értéknek (CLV).

**Kiszámított Feature-ök:**
* `recency_days`: Utolsó vásárlás óta eltelt napok száma (a vágási ponttól visszaszámolva).
* `frequency`: Sikeres (pozitív) vásárlási tranzakciók (Invoice) száma.
* `monetary_total`: Nettó elköltött összeg (a visszaküldések értékével csökkentve).
* `monetary_avg`: Átlagos nettó kosárérték ($monetary\_total / frequency$).
* `return_count`: Visszaküldött (sztornó) rendelések száma.
* `return_ratio`: Visszaküldések aránya az összes aktivitáshoz képest.

*Kritikus ML Best Practice:* Minden aggregációt szigorúan a `df_obs` adathalmazon hajtunk végre, így a modell semmilyen információt nem kap a jövőbeli (vágási pont utáni) viselkedésről.

In [ ]:
# ============================================================
# 2.2 - Kiterjesztett RFM metrikák kiszámítása
# ============================================================

print("Ügyfélszintű RFM és Return feature-ök kiszámítása a megfigyelési ablakból...\n")

# 2.2.1. Pozitív tranzakciók (Vásárlások) szűrése a Recency és Frequency számításhoz
purchases = df_obs[df_obs['Quantity'] > 0]

rfm = purchases.groupby('Customer ID').agg(
    # Recency: Cutoff dátum - utolsó vásárlás dátuma
    recency_days=('InvoiceDate', lambda x: (CUTOFF_DATE - x.max()).days),
    # Frequency: Egyedi számlák (Invoice-ok) száma
    frequency=('Invoice', 'nunique')
)

# 2.2.2. Monetary: Nettó költés (Vásárlások + Sztornók együttes összege az ablakban)
monetary = df_obs.groupby('Customer ID')['LineTotal'].sum().rename('monetary_total')
rfm = rfm.join(monetary)

# 2.2.3. Visszaküldések (Returns) azonosítása és számolása
returns = df_obs[df_obs['Quantity'] < 0]
return_counts = returns.groupby('Customer ID')['Invoice'].nunique().rename('return_count')

# Balra csatlakozás (Left join): akinek nincs visszaküldése, annál a NaN-t 0-ra cseréljük
rfm = rfm.join(return_counts).fillna({'return_count': 0})

# 2.2.4. Származtatott (Derived) mutatók kiszámítása
rfm['monetary_avg'] = rfm['monetary_total'] / rfm['frequency']
rfm['return_ratio'] = rfm['return_count'] / (rfm['frequency'] + rfm['return_count'])

# 2.2.5. QA: Extrém esetek szűrése (pl. aki a megfigyelési ablakban összességében mínuszban van)
# (Ez előfordulhat, ha valaki csak visszaküldött az X ablakban)
rfm = rfm[rfm['monetary_total'] > 0]

print("Feature Engineering sikeresen befejeződött.")
print(f"Létrejött RFM mátrix dimenziói: {rfm.shape[0]:,} ügyfél, {rfm.shape[1]} feature")
print("-" * 50)
display(rfm.head())

## 3. Statisztikai Outlier-kezelés és skálázás

A Feature Engineering során létrehozott RFM változók jellemzően erősen jobbra ferde (right-skewed) eloszlást mutatnak: a vásárlók nagy része kis értékben és ritkán vásárol, míg egy szűk réteg (a "Bálnák") extrém magas frekvenciát és költést produkál.

Mivel a következő lépésben K-means klaszterezést alkalmazunk – amely az Euklideszi távolságokra épül, és így rendkívül érzékeny a szélsőértékekre és a léptékkülönbségekre –, elengedhetetlen a változók eloszlásának vizsgálata, normalizálása és skálázása az algoritmus betanítása előtt.

### 3.1 Az eloszlások vizuális diagnosztikája
Első lépésként megvizsgáljuk az alapvető RFM mutatók (Recency, Frequency, Monetary) eloszlását boxplot ábrák segítségével, hogy felmérjük a statisztikai outlierek mértékét.

In [ ]:
# ============================================================
# 3.1 - RFM változók eloszlásának vizsgálata
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print("RFM változók eloszlásának vizualizálása...")

# Útvonal definiálása a biztonságos futtatáshoz
IMAGES_DIR = Path(".") / "assets" / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Csak az alap RFM feature-öket nézzük a diagnosztikához
features_to_plot = ['recency_days', 'frequency', 'monetary_total']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RFM Változók Eloszlása (Nyers adatok)', fontsize=16)

for i, col in enumerate(features_to_plot):
    sns.boxplot(x=rfm[col], ax=axes[i], color='skyblue')
    axes[i].set_title(f'{col} (Boxplot)')
    axes[i].set_xlabel('')

plt.tight_layout()

# Kép mentése dokumentációs céllal
fig_path_rfm = IMAGES_DIR / "rfm_raw_distributions.png"
plt.savefig(fig_path_rfm, bbox_inches="tight")
print(f"Ábra mentve: {fig_path_rfm}")
plt.show()

# Gyors statisztika a ferdeségről (Skewness)
# Ha az érték > 1 vagy < -1, az eloszlás erősen ferde
print("\nVáltozók ferdesége (Skewness):")
display(rfm[features_to_plot].skew().to_frame(name='Skewness').round(2))

### 3.2 Log-transzformáció és standardizálás

Az extrém magas ferdeségi (Skewness) értékek miatt a K-means klaszterezés előtt az alábbi kétlépcsős adat-előkészítést hajtjuk végre:
1. **Log-transzformáció (`np.log1p`):** Megtartjuk a legértékesebb vásárlóinkat ("Bálnák"), de a logaritmikus skálázással "összenyomjuk" a kiugró értékeket, így az eloszlás közelebb kerül a normál eloszláshoz. A `log1p` (log(x+1)) használata azért biztonságos, mert kezeli a 0 értékeket is (pl. 0 napos recency).
2. **Standardizálás (`StandardScaler`):** Mivel a K-means Euklideszi távolságot számol, a változókat azonos dimenzióba (átlag = 0, szórás = 1) kell hozni. Ennek hiányában a nagyobb számosságú metrikák (pl. Monetary) elnyomnák a kisebbeket (pl. Frequency).

*Megjegyzés: A klaszterezéshez csak a hagyományos R, F, M változókat használjuk. A visszaküldési (return) metrikák később, az XGBoost modellnél kapnak főszerepet.*

In [ ]:
# ============================================================
# 3.2. - Log-transzformáció és Skálázás
# ============================================================
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

print("Log-transzformáció és standardizálás folyamatban...")

# Csak az R, F, M változókat klaszterezzük
rfm_features = ['recency_days', 'frequency', 'monetary_total']
rfm_cluster_data = rfm[rfm_features].copy()

# 3.2.1. Lépés: Log-transzformáció a ferdeség csökkentésére
rfm_log = np.log1p(rfm_cluster_data)

# Nézzük meg, mennyit javult a ferdeség (Skewness)
print("\nFerdeség (Skewness) a Log-transzformáció UTÁN:")
display(rfm_log.skew().to_frame(name='Skewness').round(2))

# 3.2.2. Lépés: Standardizálás
scaler = StandardScaler()
rfm_scaled_array = scaler.fit_transform(rfm_log)

# Visszaalakítjuk DataFrame-be, hogy megmaradjon a Customer ID index
rfm_scaled = pd.DataFrame(
    rfm_scaled_array, 
    index=rfm_log.index, 
    columns=rfm_features
)

# 3.2.3. Lépés: A Scaler objektum elmentése (Kritikus lépés a Streamlit miatt!)
# Biztosítjuk, hogy a models/ mappa létezzen
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

scaler_path = MODELS_DIR / "scaler_rfm.joblib"
joblib.dump(scaler, scaler_path)

print("-" * 50)
print(f"✔️ StandardScaler sikeresen illesztve és mentve ide: {scaler_path}")
print("A skálázott (K-means bemeneti) adatok első 5 sora:")
display(rfm_scaled.head())

# 4. K-means Klaszterezés

# TODO: 6. XGBoost célváltozó (Time-window)

In [ ]:
# ============================================================
# 6.x. ...TODOs...
# ============================================================

import matplotlib.pyplot as plt

# Nézzük meg a tranzakciók napi eloszlását
daily_revenue = df.groupby(df['InvoiceDate'].dt.date)['LineTotal'].sum()

plt.figure(figsize=(15, 5))
daily_revenue.plot(title="Napi bevétel a vizsgált időszakban", color="#1f77b4")
plt.axvline(pd.to_datetime('2011-09-09'), color='red', linestyle='--', label="Tervezett Cutoff (y ablak kezdete)")
plt.ylabel("Nettó Bevétel (GBP)")
plt.legend()
plt.grid(True, alpha=0.3)

# Mentés az assets/images mappába
fig_path = IMAGES_DIR / "daily_revenue_check.png"
plt.savefig(fig_path, bbox_inches="tight")
print(f"Ábra sikeresen mentve ide: {fig_path}")

# Mennyi adat (bevétel és tranzakció) esik a célablakba?
cutoff_date = pd.to_datetime('2011-09-09')
target_window = df[df['InvoiceDate'] >= cutoff_date]

print(f"A célablakba eső tranzakciósorok száma: {len(target_window):,}")
print(f"A célablakba eső nettó bevétel: £ {target_window['LineTotal'].sum():,.2f}")
print(f"Egyedi vásárlók a célablakban: {target_window['Customer ID'].nunique():,}")